# Sample frame
**Ballon d'Or 2025 + consensus top-100 (Guardian · ESPN · FourFourTwo)** → `sample_frame.csv`.

In [1]:
import os
BASE = os.path.abspath('dissertation_data')
os.makedirs(BASE, exist_ok=True)

In [2]:
# Configuration
GUARDIAN_CSV    = os.path.join(BASE, 'guardian_top100_2025.csv')
ESPN_CSV        = os.path.join(BASE, 'espn_fc100_2025.csv')
FOURFOURTWO_CSV = os.path.join(BASE, 'fourfourtwo_top100_2025.csv')
FRAME_CSV       = os.path.join(BASE, 'sample_frame.csv')

In [3]:
# Helpers
import re, unicodedata
import pandas as pd, numpy as np
def norm_name(name):
    if not isinstance(name, str): return ''
    s = unicodedata.normalize('NFKD', name).encode('ascii','ignore').decode()
    s = re.sub(r'[^a-zA-Z ]', '', s).strip().lower()
    return re.sub(r'\s+', ' ', s)
def classify_position(raw):
    if not isinstance(raw, str) or not raw.strip(): return None
    s = re.split(r'[,/]', raw.strip().lower())[0].strip()
    if s=='gk' or 'goalkeep' in s or 'keeper' in s: return 'Goalkeeper'
    if 'midfield' in s or s in ('mf','cm','dm','am','cdm','cam'): return 'Midfielder'
    if 'back' in s or 'defen' in s or s=='df' or s in ('cb','rb','lb','rwb','lwb'): return 'Defender'
    if ('forward' in s or 'wing' in s or 'strik' in s or 'attack' in s or s=='fw'
            or s in ('cf','ss','rw','lw','st')): return 'Forward'
    return None

## Ballon d'Or 2025 (official France Football ranking — embedded & cited)

In [4]:
BALLONDOR_2025 = [
    (1,'Ousmane Dembélé','France','Forward','Paris Saint-Germain'),(2,'Lamine Yamal','Spain','Forward','Barcelona'),
    (3,'Vitinha','Portugal','Midfielder','Paris Saint-Germain'),(4,'Mohamed Salah','Egypt','Forward','Liverpool'),
    (5,'Raphinha','Brazil','Forward','Barcelona'),(6,'Achraf Hakimi','Morocco','Defender','Paris Saint-Germain'),
    (7,'Kylian Mbappé','France','Forward','Real Madrid'),(8,'Cole Palmer','England','Forward','Chelsea'),
    (9,'Gianluigi Donnarumma','Italy','Goalkeeper','Paris Saint-Germain'),(10,'Nuno Mendes','Portugal','Defender','Paris Saint-Germain'),
    (11,'Pedri','Spain','Midfielder','Barcelona'),(12,'Khvicha Kvaratskhelia','Georgia','Forward','Paris Saint-Germain'),
    (13,'Harry Kane','England','Forward','Bayern Munich'),(14,'Désiré Doué','France','Midfielder','Paris Saint-Germain'),
    (15,'Viktor Gyökeres','Sweden','Forward','Sporting CP'),(16,'Vinícius Júnior','Brazil','Forward','Real Madrid'),
    (17,'Robert Lewandowski','Poland','Forward','Barcelona'),(18,'Scott McTominay','Scotland','Midfielder','Napoli'),
    (19,'João Neves','Portugal','Midfielder','Paris Saint-Germain'),(20,'Lautaro Martínez','Argentina','Forward','Inter Milan'),
    (21,'Serhou Guirassy','Guinea','Forward','Borussia Dortmund'),(22,'Alexis Mac Allister','Argentina','Midfielder','Liverpool'),
    (23,'Jude Bellingham','England','Midfielder','Real Madrid'),(24,'Fabián Ruiz','Spain','Midfielder','Paris Saint-Germain'),
    (25,'Denzel Dumfries','Netherlands','Defender','Inter Milan'),(26,'Erling Haaland','Norway','Forward','Manchester City'),
    (27,'Declan Rice','England','Midfielder','Arsenal'),(28,'Virgil van Dijk','Netherlands','Defender','Liverpool'),
    (29,'Florian Wirtz','Germany','Midfielder','Bayer Leverkusen'),(30,'Michael Olise','France','Forward','Bayern Munich'),
]
ballondor = pd.DataFrame(BALLONDOR_2025, columns=['bdor_rank','player_name','nationality','position','club'])
ballondor['name_key'] = ballondor['player_name'].map(norm_name)
print('Ballon d\'Or:', len(ballondor), 'players')

Ballon d'Or: 30 players


## Consensus from the three media lists
Upload all 3 top 100 list in dissertation data folder first

In [5]:
def need(p):
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing {os.path.basename(p)} — put it in {BASE} and re-run.')
def load_ranked(path, rc='rank'):
    need(path); df = pd.read_csv(path); df['name_key'] = df['player_name'].map(norm_name)
    if rc in df.columns: df[rc] = pd.to_numeric(df[rc], errors='coerce')
    return df
guardian, fourfourtwo, espn = load_ranked(GUARDIAN_CSV), load_ranked(FOURFOURTWO_CSV), load_ranked(ESPN_CSV,'none')
keys = pd.Index(pd.concat([guardian['name_key'],fourfourtwo['name_key'],espn['name_key']]).unique())
cons = pd.DataFrame({'name_key': keys})
names = pd.concat([guardian[['name_key','player_name']],fourfourtwo[['name_key','player_name']],
                   espn[['name_key','player_name']]]).drop_duplicates('name_key')
cons = cons.merge(names, on='name_key', how='left')
cons['guardian_rank']=cons['name_key'].map(guardian.set_index('name_key')['rank'])
cons['fourfourtwo_rank']=cons['name_key'].map(fourfourtwo.set_index('name_key')['rank'])
cons['espn_listed']=cons['name_key'].isin(set(espn['name_key'])).astype(int)
cons['consensus_count']=(cons['guardian_rank'].notna().astype(int)+cons['fourfourtwo_rank'].notna().astype(int)+cons['espn_listed'])
cons['avg_rank']=cons[['guardian_rank','fourfourtwo_rank']].mean(axis=1)
consensus=cons.sort_values(['consensus_count','avg_rank'],ascending=[False,True]).reset_index(drop=True)
print('Consensus pool:', len(consensus), 'players')

Consensus pool: 146 players


In [7]:
frame=(pd.concat([ballondor[['name_key','player_name']],consensus[['name_key','player_name']]],
                 ignore_index=True).drop_duplicates('name_key').reset_index(drop=True))
frame['in_ballondor']=frame['name_key'].isin(set(ballondor['name_key']))
frame['consensus_count']=frame['name_key'].map(consensus.set_index('name_key')['consensus_count']).fillna(0).astype(int)
bd=ballondor.set_index('name_key')
frame['bdor_club']=frame['name_key'].map(bd['club']); frame['bdor_nationality']=frame['name_key'].map(bd['nationality'])
frame.to_csv(FRAME_CSV,index=False)
print('Sample frame:',len(frame),'players ->',FRAME_CSV)
frame.head()

Sample frame: 146 players -> /content/dissertation_data/sample_frame.csv


,name_key,player_name,in_ballondor,consensus_count,bdor_club,bdor_nationality
0,ousmane dembele,Ousmane Dembélé,True,3,Paris Saint-Germain,France
1,lamine yamal,Lamine Yamal,True,3,Barcelona,Spain
2,vitinha,Vitinha,True,3,Paris Saint-Germain,Portugal
3,mohamed salah,Mohamed Salah,True,3,Liverpool,Egypt
4,raphinha,Raphinha,True,3,Barcelona,Brazil
